# Assignment 10: Deep Learning with PyTorch

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

If PyTorch is not installed, run `%pip install torch` in a cell, or use Google Colab (PyTorch is already installed there).

## Setup

In Assignment 9, scikit-learn trained the network for us. Here we use **PyTorch**, the library behind most deep learning, and write the training loop ourselves.

The data is scikit-learn's handwritten digits: 1,797 images of size 8×8, with labels 0–9. No download is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
digits = load_digits()
X = digits.images / 16.0            # pixel values 0-16  ->  0-1
y = digits.target

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, img, label in zip(axes, X, y):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()
print(X.shape)

### Question 1. Prepare the data

a. Split the data into 60% training, 20% validation and 20% test (`random_state=1`, stratified by `y`).

b. Convert each part to PyTorch tensors: images as `float32` and **flattened** to 64 numbers, labels as `long` (integers).

c. Wrap the training data in a `DataLoader` with `batch_size=32` and `shuffle=True`.

In [ ]:
# TODO: split into train/validation/test

def to_tensors(X, y):
    # TODO: return a flattened float32 tensor of images and a long tensor of labels

Xtr, ytr = to_tensors(X_train, y_train)
Xva, yva = to_tensors(X_val, y_val)
Xte, yte = to_tensors(X_test, y_test)

# TODO: create train_loader
print(Xtr.shape, Xva.shape, Xte.shape)

### Question 2. Define the network

Complete the class below: a network with two hidden layers of 128 and 64 neurons, ReLU activations, and an output layer with 10 neurons (one score per digit).

There is no softmax at the end: `nn.CrossEntropyLoss` applies it for us.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        # TODO: define self.net as an nn.Sequential with the layers described above (put nn.Dropout(dropout) after each ReLU)

    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

### Question 3. Write the training loop

Complete `train_model`. In every epoch it goes through all the mini-batches; for each batch it must

1. compute the model's outputs,
2. compute the loss,
3. reset the old gradients,
4. backpropagate,
5. update the weights.

After each epoch it records the training and validation loss, so we can plot learning curves.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        out = model(X)
        loss = loss_fn(out, y).item()
        acc = (out.argmax(dim=1) == y).float().mean().item()
    model.train()
    return loss, acc

def train_model(model, loader, epochs=30, lr=0.1, weight_decay=0.0, adam=False):
    opt_class = torch.optim.Adam if adam else torch.optim.SGD
    optimizer = opt_class(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        for xb, yb in loader:
            # TODO: the five steps of one training step
        history["train"].append(evaluate(model, Xtr, ytr)[0])
        history["val"].append(evaluate(model, Xva, yva)[0])
    return history

def plot_history(history, title):
    plt.plot(history["train"], label="training loss")
    plt.plot(history["val"], label="validation loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(title); plt.legend()

torch.manual_seed(0)
model = MLP()
history = train_model(model, train_loader, epochs=30, lr=0.1)
plot_history(history, "MLP, learning rate 0.1"); plt.show()
print("validation accuracy: %.3f" % evaluate(model, Xva, yva)[1])

### Question 4. Vocabulary check

With 1,078 training images and `batch_size=32`:

a. How many **mini-batches** (weight updates) are there in one **epoch**?

b. How many weight updates happen in 30 epochs?

c. Why do we train on mini-batches instead of the whole training set at once, or one image at a time?

**Your answer:** *Write your answer here.*

### Question 5. The learning rate

Train a fresh `MLP()` for 30 epochs with each learning rate in `[0.001, 0.01, 0.1, 1.0]`. Plot the four **validation** loss curves on one graph and report each final validation accuracy. Which learning rate would you choose, and what goes wrong with the others?

In [ ]:
plt.figure(figsize=(8, 4.5))
for lr in [0.001, 0.01, 0.1, 1.0]:
    torch.manual_seed(0)
    # TODO: train a fresh MLP with this learning rate, plot its validation loss and print its validation accuracy
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.yscale("log"); plt.legend(); plt.show()

**Your answer:** *Write your answer here.*

### Question 6. Overfitting and how to fight it

Deep networks have many parameters and can memorize a small training set. To see this, we train on only **150** training images for 300 epochs. We also switch to the **Adam** optimizer (`adam=True`, `lr=0.001`), the most widely used optimizer in deep learning; it adapts the step size for each weight and memorizes faster than plain SGD.

a. Train `MLP()` (no regularization) and plot its learning curves. Describe what happens to the training loss and to the validation loss.

b. Train `MLP(dropout=0.3)` with `weight_decay=1e-3` and plot its curves.

c. **Early stopping** keeps the model from the epoch with the lowest validation loss. At which epoch would early stopping have stopped each model? How much lower is the validation loss at that epoch than at the end?

d. Compare the final validation accuracies too. Did dropout and weight decay make a big difference here? Which of the three tools (dropout, weight decay, early stopping) helped the most on this data?

In [ ]:
small_loader = DataLoader(TensorDataset(Xtr[:150], ytr[:150]), batch_size=32, shuffle=True)

def best_and_final(history):
    best_epoch = int(np.argmin(history["val"]))
    return best_epoch, history["val"][best_epoch], history["val"][-1]

plt.figure(figsize=(12, 4))
for i, (name, dropout, wd) in enumerate([("No regularization", 0.0, 0.0),
                                         ("Dropout 0.3 + weight decay 1e-3", 0.3, 1e-3)]):
    torch.manual_seed(0)
    # TODO: train MLP(dropout) on small_loader for 300 epochs with Adam, lr=0.001 and this weight decay
    plt.subplot(1, 2, i + 1); plot_history(h, name); plt.yscale("log")
    best_epoch, best_loss, final_loss = best_and_final(h)
    print(f"{name:<34} lowest val loss {best_loss:.3f} at epoch {best_epoch}, final val loss {final_loss:.3f}, "
          f"final val accuracy {evaluate(m, Xva, yva)[1]:.3f}")
plt.show()

**Your answer:** *Write your answer here.*

### Question 7. Final test

Retrain your best setup from Questions 5–6 on the **full** training set, then report its accuracy on the **test** set. Why did we wait until now to use the test set?

In [ ]:
torch.manual_seed(0)
# TODO: train your final model on train_loader and call it `final`
print("test accuracy: %.3f" % evaluate(final, Xte, yte)[1])

**Your answer:** *Write your answer here.*